In [ ]:
import json
import pandas as pd
import torch
import numpy as np
from tqdm import tqdm
from accelerate.hooks import ModelHook, remove_hook_from_module
from accelerate.hooks import add_hook_to_module, remove_hook_from_module
from PIL import Image
from utils import *
from text_utils import *
from transformers import (
    CLIPVisionModel, CLIPImageProcessor,
    ViTModel, ViTForImageClassification, ViTImageProcessor,
    ViTMAEForPreTraining, AutoImageProcessor,
    AutoProcessor, LlavaForConditionalGeneration,
    Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor,
    SamModel, SamProcessor, AutoModel, Gemma3ForConditionalGeneration,AutoConfig
)


In [ ]:

torch.cuda.empty_cache()
name = "Qwen/Qwen2.5-VL-3B-Instruct"
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    name,
    torch_dtype=torch.bfloat16,
).to("cuda")
processor = AutoProcessor.from_pretrained(name)

def collect_coeffs(file_path):
    # Collect R² values from cluster files
    with open(file_path,"r") as file:
        data = json.load(file)
    df = pd.DataFrame.from_dict(data,orient="index")
    #identify last layer index
    last_layer = df.index[-1]
    last_layer_data = data[last_layer]
    last_layer_data["layer"]=int(last_layer)
    #get absolute coefficient value
    last_layer_data["absolute_latitude"] = np.absolute(last_layer_data["latitude_coefficients"])
    last_layer_data["absolute_longitude"] = np.absolute(last_layer_data["longitude_coefficients"])
    #negate the array to sort descending order
    last_layer_data["latitude_indexes"]= np.argsort(-last_layer_data["absolute_latitude"])
    last_layer_data["longitude_indexes"]= np.argsort(-last_layer_data["absolute_longitude"])
    norm_lat = (last_layer_data["absolute_latitude"]-np.min(last_layer_data["absolute_latitude"]))/(np.max(last_layer_data["absolute_latitude"])-np.min(last_layer_data["absolute_latitude"]))
    norm_lon = (last_layer_data["absolute_longitude"]-np.min(last_layer_data["absolute_longitude"]))/(np.max(last_layer_data["absolute_longitude"])-np.min(last_layer_data["absolute_longitude"]))
    last_layer_data["combined_indexes"] = np.argsort(-(norm_lat+norm_lon))

    #identify first layer index
    first_layer = df.index[0]
    first_layer_data = data[first_layer]
    first_layer_data["layer"]=int(first_layer)
    #get absolute coefficient value
    first_layer_data["absolute_latitude"] = np.absolute(first_layer_data["latitude_coefficients"])
    first_layer_data["absolute_longitude"] = np.absolute(first_layer_data["longitude_coefficients"])
    #negate the array to sort descending order
    first_layer_data["latitude_indexes"]= np.argsort(-first_layer_data["absolute_latitude"])
    first_layer_data["longitude_indexes"]= np.argsort(-first_layer_data["absolute_longitude"])
    norm_lat = (first_layer_data["absolute_latitude"]-np.min(first_layer_data["absolute_latitude"]))/(np.max(first_layer_data["absolute_latitude"])-np.min(first_layer_data["absolute_latitude"]))
    norm_lon = (first_layer_data["absolute_longitude"]-np.min(first_layer_data["absolute_longitude"]))/(np.max(first_layer_data["absolute_longitude"])-np.min(first_layer_data["absolute_longitude"]))
    first_layer_data["combined_indexes"] = np.argsort(-(norm_lat+norm_lon))

    #identify best_layer_index
    best_layer = df[df["R2"]==df[:-1]["R2"].max()].index[0]
    best_layer_data = data[best_layer]
    best_layer_data["layer"]=int(best_layer)
    best_layer_data["absolute_latitude"] = np.absolute(best_layer_data["latitude_coefficients"])
    best_layer_data["absolute_longitude"] = np.absolute(best_layer_data["longitude_coefficients"])
    best_layer_data["latitude_indexes"]= np.argsort(-best_layer_data["absolute_latitude"])
    best_layer_data["longitude_indexes"]= np.argsort(-best_layer_data["absolute_longitude"])
    norm_lat = (best_layer_data["absolute_latitude"]-np.min(best_layer_data["absolute_latitude"]))/(np.max(best_layer_data["absolute_latitude"])-np.min(best_layer_data["absolute_latitude"]))
    norm_lon = (best_layer_data["absolute_longitude"]-np.min(best_layer_data["absolute_longitude"]))/(np.max(best_layer_data["absolute_longitude"])-np.min(best_layer_data["absolute_longitude"]))
    best_layer_data["combined_indexes"] = np.argsort(-(norm_lat+norm_lon))
    return last_layer_data,best_layer_data,first_layer_data
def get_inputs(image,prompt=None):
    image = torch.from_numpy(image)
    messages =[
    {

      "role": "user",
      "content": [
          {"type": "image"},
        ],
    },
    ]
    if prompt:
        messages[0]["content"].append({"type":"text","text":prompt})
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text],
        images=image.permute(2,0,1),
        padding=True,
        return_tensors="pt",
    )
    return inputs.to("cuda")
class ModifyHook(ModelHook):
    def post_forward(self, module, output):
        global gen_count
        if gen_count>=len(important_output_list):
            return output
        new_output = output[0]
        new_output[:,-1,indexes] = important_output_list[gen_count][0][:,-1,indexes]
        new_tup = (new_output,)
        gen_count+=1
        return new_tup

class RandomHook(ModelHook):
    def post_forward(self, module, output):
        global gen_count
        if gen_count>=len(important_output_list):
            return output
        new_output = output[0]
        new_output[:,-1,random_indexes] = important_output_list[gen_count][0][:,-1,random_indexes]
        new_tup = (new_output,)
        gen_count+=1
        return new_tup



class ExtractHook(ModelHook):
    def post_forward(self, module, output):
        important_output_list.append(output)
        return output
import pandas as pd
dataset = pd.read_csv("embedding_swap/metadata.csv")
dataset['img'] = dataset['file'].apply(lambda filename: np.array(Image.open('embedding_swap/pictures/' + filename)))
test_set = [(20,6),(20,7),(6,34),(37,43),(18,12),(31,43)]
np.random.seed(42)
source_figs = list(range(51))
target_figs = []
for fig in source_figs:
    aux_set = dataset[dataset["country"]!=dataset.loc[fig,"country"]]
    if fig not in [20,6,37,18,31]:
        target_figs.append(np.random.choice(list(aux_set.index),size=5,replace=False))
    elif fig==20:
        target_figs.append(np.random.choice(list(aux_set.index),size=3,replace=False))
    else:
        target_figs.append(np.random.choice(list(aux_set.index),size=4,replace=False))
for fig in source_figs:
    for target in target_figs[fig]:
        test_set.append((fig,int(target)))

from tqdm import tqdm
import json

last_layer_data,best_layer_data,first_layer_data = collect_coeffs("./results/geocells_08/Qwen2.5-VL-3B-Instruct/text_landmark_activation.json")
data = first_layer_data
layer=first_layer_data['layer']
torch.manual_seed(42)
shuffled_indices = torch.randperm(len(data["combined_indexes"]))


results = {}
for p in np.linspace(0,1,11):
    source_outputs,target_outputs,modified_outputs,random_outputs = [],[],[],[]
    frac = p
    for in_img_id,out_img_id in tqdm(test_set):
        in_img= dataset.loc[in_img_id,'img']
        out_img = dataset.loc[out_img_id,'img']
        important_output_list =[]
        
        indexes = data["combined_indexes"][:int(frac*len(data["combined_indexes"]))]
        random_indexes = shuffled_indices[:int(frac*len(shuffled_indices))]
    
        with torch.no_grad():
            inputs = get_inputs(in_img,prompt=None)
            out=model.generate(**inputs,max_new_tokens=100,temperature=0.0001)
            generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, out)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        source_outputs.append(output_text)
        
    # Attach to desired layers to obtain the output
        add_hook_to_module(model.language_model.model.layers[layer],ExtractHook())
        with torch.no_grad():
            inputs = get_inputs(out_img,prompt=None)
            out=model.generate(**inputs,max_new_tokens=100,temperature=0.0001)
            generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, out)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        remove_hook_from_module(model.language_model.model.layers[layer])
        target_outputs.append(output_text)
        
        gen_count=0
        add_hook_to_module(model.language_model.model.layers[layer],ModifyHook())
        with torch.no_grad():
            inputs = get_inputs(in_img,prompt=None)
            out=model.generate(**inputs,max_new_tokens=100,temperature=0.0001)
        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, out)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        remove_hook_from_module(model.language_model.model.layers[layer])
        modified_outputs.append(output_text)
        
        gen_count=0
        add_hook_to_module(model.language_model.model.layers[layer],RandomHook())
        with torch.no_grad():
            inputs = get_inputs(in_img,prompt=None)
            out=model.generate(**inputs,max_new_tokens=100,temperature=0.0001)
        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, out)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        remove_hook_from_module(model.language_model.model.layers[layer])
        random_outputs.append(output_text)
    results[p] = {'source':source_outputs,'target':target_outputs,'modified':modified_outputs,'random':random_outputs,'tuples':test_set}
    with open("embedding_swap/embedding_swap_generations.json","w",encoding="utf-8") as file:
        json.dump(results,file,ensure_ascii=False)


In [2]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stopwords = set(stopwords.words('english'))

def remove_stopwords(text):
    if isinstance(text,str):
        tokens = word_tokenize(text.lower())
    
        # Remove stopwords
        filtered_tokens = [word for word in tokens if word not in stopwords]
        return filtered_tokens
    elif isinstance(text,list):
        sents = []
        for string in text:
            tokens = word_tokenize(string.lower())
            filtered_tokens = [word for word in tokens if word not in stopwords]
            sents.append(filtered_tokens)
        return sents
    return ''

In [ ]:
dataset = pd.read_csv("embedding_swap/metadata.csv")
dataset = dataset.map(lambda x: x.lower() if isinstance(x, str) else '')
dataset['alternate_names'] = dataset['alternate_names'].apply(lambda x: x.split(';') if (isinstance(x,str) and len(x)>0) else [])
dataset["location_names"] = dataset['city'].apply(lambda x:[x] if x else []) + dataset['region'].apply(lambda x:[x] if x else []) + dataset['country'].apply(lambda x:[x] if x else []) 
dataset["place_names"] = dataset['name'].apply(lambda x:[x]) + dataset['alternate_names']

In [4]:
def identify_place(text, place_row):
    text=text.lower()
    if place_row['country'] in text or place_row['region'] in text or place_row['city']in text:
        return True
    elif (place_row['country']=='united states') and 'usa' in text:
        return True
    return False

In [ ]:
responses = json.load(open("embedding_swap/embedding_swap_generations.json",encoding="utf-8"))

In [ ]:
import re
import spacy

# optional but useful for robust matching:
nlp = spacy.load("en_core_web_sm")

def normalize(text):
    return re.sub(r'\s+', ' ', text.lower().strip())

def any_match(text, patterns):
    """Return True if ANY name/location from patterns is found in text."""
    text = normalize(text)
    for p in patterns:
        if p.lower() in text:
            return True
    return False


In [7]:
def detect_entities(row, places_df):
    # Here we assume each row corresponds to one PLACE ID
    # Modify if necessary
    
    source_place_row = dataset.loc[row['s_img']]
    target_place_row = dataset.loc[row['t_img']]

    source_spot_present   = any_match(row["source"],   source_place_row["place_names"])
    source_loc_present    = any_match(row["source"],   source_place_row["location_names"])

    target_spot_present   = any_match(row["target"],   target_place_row["place_names"])
    target_loc_present    = any_match(row["target"],   target_place_row["location_names"])

    return pd.Series({
        "source_spot_present": source_spot_present,
        "source_loc_present": source_loc_present,
        "target_spot_present": target_spot_present,
        "target_loc_present": target_loc_present,
    })



In [8]:
def modification_success(row, places_df):
    source_place_row = dataset.loc[row['s_img']]
    target_place_row = dataset.loc[row['t_img']]
    source_spot_present_mod   = any_match(row["modified"],   source_place_row["place_names"])
    target_loc_present_mod    = any_match(row["modified"],   target_place_row["location_names"])

    source_spot_present_random   = any_match(row["random"],   source_place_row["place_names"])
    target_loc_present_random    = any_match(row["random"],   target_place_row["location_names"])


    return pd.Series({
        "modified_success": target_loc_present_mod & source_spot_present_mod,
        "random_success": source_spot_present_random & target_loc_present_random,
        "modified_location": target_loc_present_mod,
        "modified_spot":source_spot_present_mod,
        "random_location":target_loc_present_random,
        "random_spot":source_spot_present_random
    })



In [9]:
def get_stability_efficacy(df):
    stats = {
        "mod_stability":df['modified_spot'].sum()/len(df),
        "mod_efficacy":df['modified_location'].sum()/len(df),
        "mod_success_rate":df['modified_success'].sum()/len(df),
        "rand_stability":df['random_spot'].sum()/len(df),
        "rand_efficacy":df['random_location'].sum()/len(df),
        "rand_success_rate":df['random_success'].sum()/len(df),
            }
    return stats

In [10]:
stats=[]
ps=[]
all_data = {}
for p,data in responses.items():
    df = pd.DataFrame(data)
    for col in df.columns:
        df[col]=df[col].apply(lambda x:x[0] if len(x)<2 else x)
    df['s_img']=df['tuples'].apply(lambda tup:int(tup[0]))
    df['t_img']=df['tuples'].apply(lambda tup:int(tup[1]))

    df = df.join(df.apply(lambda r: detect_entities(r, dataset), axis=1))
    print(len(df))
    df=df[df['source_spot_present'] & df["source_loc_present"] & df["target_spot_present"] & df["target_loc_present"] ]
    print(len(df))
    df = df.join(df.apply(lambda r: modification_success(r, dataset), axis=1))
    stats.append(get_stability_efficacy(df))
    p=round(float(p),1)
    ps.append(p)
    all_data[p]=df
table = pd.DataFrame(stats)
table['p'] = ps
table

255
218
255
215
255
215
255
215
255
218
255
217
255
218
255
217
255
217
255
216
255
218


,mod_stability,mod_efficacy,mod_success_rate,rand_stability,rand_efficacy,rand_success_rate,p
0,0.990826,0.000000,0.000000,0.990826,0.000000,0.000000,0.0
1,0.990698,0.000000,0.000000,0.986047,0.000000,0.000000,0.1
2,0.976744,0.000000,0.000000,0.995349,0.000000,0.000000,0.2
3,0.976744,0.000000,0.000000,0.986047,0.000000,0.000000,0.3
4,0.967890,0.000000,0.000000,0.967890,0.000000,0.000000,0.4
5,0.953917,0.000000,0.000000,0.935484,0.000000,0.000000,0.5
6,0.899083,0.000000,0.000000,0.885321,0.000000,0.000000,0.6
7,0.797235,0.000000,0.000000,0.788018,0.000000,0.000000,0.7
8,0.493088,0.110599,0.041475,0.668203,0.004608,0.004608,0.8
9,0.314815,0.439815,0.134259,0.314815,0.458333,0.120370,0.9
